<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h1>بازبینی نهایی مؤلف — پایلوت دیتاست معیار پروژه RAG Finance</h1>

<p>
این Notebook برای بازبینی علمی و معنایی <strong>۲۲ نمونه پایلوت</strong> از «دیتاست معیار» پروژه ساخته شده است.
دیتاست معیار یا <code>Golden Dataset</code> مجموعه‌ای از سؤال‌های مرجع است که برای هر سؤال،
پاسخ مورد انتظار و شواهد دقیق منبع مشخص می‌شود. در مراحل بعدی پروژه، کیفیت جست‌وجو، بازیابی،
تولید پاسخ و استناد سامانه با همین نمونه‌های مرجع سنجیده می‌شود.
</p>

<h3>چرا نظر مؤلف لازم است؟</h3>

<p>
کنترل‌های فنی می‌توانند بررسی کنند که منبع وجود دارد، شناسه‌ها درست‌اند و پاسخ به شواهد متصل است؛
اما تشخیص اینکه برداشت نهایی از متن دقیقاً با منظور علمی مؤلف سازگار است، نیاز به بازبینی انسانی مؤلف دارد.
به همین دلیل این ۲۲ نمونه، پس از بازبینی فنی و انسانی اولیه، برای نظر نهایی دکتر کمیل رودی آماده شده‌اند.
</p>

<h3>در هر مورد چه چیزی دیده می‌شود؟</h3>

<ul>
<li><strong>متن و شواهد منبع</strong> که سؤال و پاسخ از آن ساخته شده است.</li>
<li><strong>سؤال نهایی پیشنهادی</strong> که قرار است به‌عنوان سؤال مرجع استفاده شود.</li>
<li><strong>پاسخ نهایی پیشنهادی</strong> که باید مستقیماً از شواهد منبع پشتیبانی شود.</li>
<li><strong>وضعیت استناد</strong> و در صورت وجود، مشخصات فنی محل استناد.</li>
<li><strong>وضعیت بازبینی مؤلف</strong> که در شروع این Notebook هنوز ثبت نشده است.</li>
</ul>

<h3>از مؤلف چه می‌خواهیم؟</h3>

<p>
برای هر مورد فقط این چهار موضوع بررسی شود:
آیا سؤال طبیعی و درست است؟
آیا پاسخ با منظور واقعی منبع سازگار است؟
آیا نکته‌ای بیش از منبع به پاسخ اضافه نشده است؟
و آیا شواهد نمایش‌داده‌شده برای پشتیبانی پاسخ کافی هستند؟
اگر موردی نیاز به اصلاح دارد، اصلاح پیشنهادی مؤلف باید جداگانه ثبت شود.
</p>

<h3>محدوده این Notebook</h3>

<p>
این Notebook <strong>فقط Viewer و ابزار بازبینی Read-Only</strong> است.
هیچ متن منبع، سؤال، پاسخ، Registry یا Artifact پروژه را تغییر نمی‌دهد.
ثبت رسمی تصمیم‌های مؤلف باید در Workflow مجاز پروژه انجام شود تا Lineage و تاریخچه تصمیم‌ها حفظ شود.
</p>

<h3>شرط Done / PASS</h3>

<p>
Notebook زمانی از نظر فنی آماده بازبینی مؤلف است که دقیقاً ۲۲ Case کامل و یکتا از Registry خوانده شود،
برای هر ۲۲ مورد سؤال نهایی، پاسخ نهایی و شواهد منبع قابل نمایش باشند،
Queue پایلوت صفر باشد، و Final Health بدون خطا PASS شود.
</p>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>پیش‌بررسی Repository و داده‌های بازبینی</h2>

<p>
پیش از نمایش Caseها، این Notebook به‌صورت Read-Only فایل
<code>human_review_registry.json</code> را در Repository واقعی پیدا می‌کند و وضعیت آن را بررسی می‌کند.
هدف این مرحله جلوگیری از نمایش داده قدیمی، ناقص یا اشتباه است.
</p>

<p>
این Preflight باید ثابت کند که Registry قابل خواندن است، دقیقاً ۲۲ Pilot تکمیل‌شده و یکتا وجود دارد،
هیچ Pilot فعالی باقی نمانده، Remaining Queue برابر صفر است و ۲۲ Case برای Author Review آماده‌اند.
در این مرحله هیچ فایلی نوشته یا تغییر داده نمی‌شود.
</p>

</div>

In [1]:
# Code Cell 01

from pathlib import Path
import hashlib
import html
import json
import re

from IPython.display import HTML, display

In [2]:
# Code Cell 02

EXPECTED_CASE_COUNT = 22


def find_registry_path():
    cwd = Path.cwd().resolve()

    candidates = [
        cwd / "data" / "phase2" / "human_review" / "human_review_registry.json",
        cwd / ".." / ".." / "data" / "phase2" / "human_review" / "human_review_registry.json",
    ]

    for parent in [cwd, *cwd.parents]:
        candidates.append(
            parent
            / "data"
            / "phase2"
            / "human_review"
            / "human_review_registry.json"
        )

    checked = set()

    for candidate in candidates:
        candidate = candidate.resolve()
        key = str(candidate)

        if key in checked:
            continue

        checked.add(key)

        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        "human_review_registry.json was not found."
    )


REGISTRY_PATH = find_registry_path()

review_registry = json.loads(
    REGISTRY_PATH.read_text(
        encoding="utf-8"
    )
)

completed_cases = review_registry.get(
    "completed_human_review_cases",
    []
)

author_review = review_registry.get(
    "author_review",
    {}
) or {}

author_ready_cases = author_review.get(
    "cases_ready_for_review",
    []
) or []

completed_ids = [
    case.get("pilot_case_id")
    for case in completed_cases
    if isinstance(case, dict)
]

preflight_checks = {
    "registry_is_dict": isinstance(review_registry, dict),
    "completed_cases_is_list": isinstance(completed_cases, list),
    "completed_case_count_22": len(completed_cases) == EXPECTED_CASE_COUNT,
    "completed_ids_unique_22": (
        len(completed_ids) == EXPECTED_CASE_COUNT
        and len(set(completed_ids)) == EXPECTED_CASE_COUNT
    ),
    "remaining_pilots_zero": review_registry.get("remaining_pilot_count") == 0,
    "active_workspace_none": review_registry.get("active_workspace") is None,
    "author_ready_count_22": len(author_ready_cases) == EXPECTED_CASE_COUNT,
}

preflight_gate = all(preflight_checks.values())

print("Registry:", REGISTRY_PATH)
print("Completed Pilot cases:", len(completed_cases))
print("Unique Pilot IDs:", len(set(completed_ids)))
print("Cases ready for Author Review:", len(author_ready_cases))
print("Remaining Pilots:", review_registry.get("remaining_pilot_count"))
print("Active Workspace:", review_registry.get("active_workspace"))
print("Author Final Review:", author_review.get("final_review_status"))

print("\nPRE-FLIGHT CHECKS")
for name, status in preflight_checks.items():
    print(f"{name}:", status)

print(
    "\nCODE CELL 02 GATE:",
    "PASS" if preflight_gate else "FAIL",
)

assert preflight_gate, (
    "Author Review preflight failed. "
    "Do not continue until the Registry state is corrected."
)

Registry: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Completed Pilot cases: 22
Unique Pilot IDs: 22
Cases ready for Author Review: 22
Remaining Pilots: 0
Active Workspace: None
Author Final Review: NOT_STARTED

PRE-FLIGHT CHECKS
registry_is_dict: True
completed_cases_is_list: True
completed_case_count_22: True
completed_ids_unique_22: True
remaining_pilots_zero: True
active_workspace_none: True
author_ready_count_22: True

CODE CELL 02 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>آماده‌سازی ۲۲ Case برای نمایش مؤلف</h2>

<p>
Registry در طول ساخت پایلوت‌ها بیش از یک Schema تاریخی داشته است.
در نسخه قدیمی، سؤال نهایی داخل تاریخچه Draftها ذخیره می‌شد؛
در نسخه جدید، سؤال داخل <code>question_review</code> قرار دارد.
Viewer قبلی فقط Schema قدیمی را می‌خواند و به همین دلیل سؤال بسیاری از Caseهای جدید نمایش داده نمی‌شد.
</p>

<p>
این مرحله بدون تغییر Registry، هر دو Schema را به یک نمای واحد تبدیل می‌کند.
برای هر Case، سؤال نهایی، پاسخ نهایی، شواهد اصلی و تکمیلی، Citation و وضعیت Author Review استخراج می‌شود.
هیچ داده‌ای بازنویسی نمی‌شود و هیچ حدسی جای Evidence را نمی‌گیرد.
</p>

</div>

In [3]:
# Code Cell 03

def is_accept(value):
    return (
        isinstance(value, str)
        and value.strip().upper() == "ACCEPT"
    )


def first_text(mapping, keys):
    if not isinstance(mapping, dict):
        return None

    for key in keys:
        value = mapping.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

    return None


def final_question(case):
    question_review = case.get("question_review") or {}

    question = first_text(
        question_review,
        [
            "question_text",
            "final_question_text",
            "accepted_question_text",
        ],
    )

    if question:
        return question, "question_review"

    drafts = case.get("model_drafts") or []

    accepted_drafts = [
        draft
        for draft in drafts
        if (
            isinstance(draft, dict)
            and is_accept(draft.get("human_decision"))
            and isinstance(draft.get("question_text"), str)
            and draft.get("question_text").strip()
        )
    ]

    if accepted_drafts:
        return (
            accepted_drafts[-1]["question_text"].strip(),
            "accepted_model_draft",
        )

    valid_drafts = [
        draft
        for draft in drafts
        if (
            isinstance(draft, dict)
            and isinstance(draft.get("question_text"), str)
            and draft.get("question_text").strip()
        )
    ]

    if valid_drafts:
        return (
            valid_drafts[-1]["question_text"].strip(),
            "latest_model_draft",
        )

    return None, None


def final_answer(case):
    answer_review = case.get("expected_answer_review") or {}

    answer = first_text(
        answer_review,
        [
            "expected_answer_text",
            "answer_text",
            "final_answer_text",
        ],
    )

    if answer:
        return answer

    completion_summary = case.get("completion_summary") or {}

    return first_text(
        completion_summary,
        [
            "expected_answer_text",
            "answer_text",
        ],
    )


def collect_evidence_records(case):
    primary = (
        case.get("source_evidence")
        or case.get("source_evidence_snapshot")
        or []
    )

    supporting = (
        case.get("supporting_book_evidence")
        or case.get("supporting_evidence")
        or []
    )

    citation_review = case.get("citation_review") or {}
    citation_record = (
        citation_review.get("citation_mapping_record")
        or {}
    )
    citation_entries = (
        citation_record.get("citation_entries")
        or []
    )

    def normalize_records(records):
        if isinstance(records, dict):
            records = [records]

        if not isinstance(records, list):
            return []

        return [
            record
            for record in records
            if (
                isinstance(record, dict)
                and isinstance(record.get("text"), str)
                and record.get("text").strip()
            )
        ]

    primary_records = normalize_records(primary)
    supporting_records = normalize_records(supporting)
    citation_records = normalize_records(citation_entries)

    seen = set()
    combined = []

    for group_name, records in [
        ("primary", primary_records),
        ("supporting", supporting_records),
        ("citation", citation_records),
    ]:
        for record in records:
            text = record["text"].strip()
            key = hashlib.sha256(
                text.encode("utf-8")
            ).hexdigest()

            if key in seen:
                continue

            seen.add(key)

            item = dict(record)
            item["_viewer_group"] = group_name
            combined.append(item)

    return combined


def citation_status(case):
    citation_review = case.get("citation_review") or {}

    decision = (
        citation_review.get("human_decision")
        or citation_review.get("decision")
        or citation_review.get("human_adjudication_status")
    )

    citation_record = (
        citation_review.get("citation_mapping_record")
        or {}
    )

    citation_id = (
        citation_record.get("citation_mapping_id")
        or citation_review.get("citation_mapping_id")
    )

    return decision, citation_id


def author_status(case):
    review = case.get("author_final_review") or {}

    return (
        review.get("status")
        or review.get("decision")
        or "NOT_STARTED"
    )


author_cases = []
normalization_issues = []


for index, case in enumerate(
    completed_cases,
    start=1,
):
    question, question_source = final_question(case)
    answer = final_answer(case)
    evidence = collect_evidence_records(case)
    citation_decision, citation_id = citation_status(case)

    normalized = {
        "number": index,
        "pilot_case_id": case.get("pilot_case_id"),
        "question": question,
        "question_source": question_source,
        "answer": answer,
        "evidence": evidence,
        "citation_decision": citation_decision,
        "citation_mapping_id": citation_id,
        "author_status": author_status(case),
        "raw_case": case,
    }

    author_cases.append(normalized)

    missing = []

    if not normalized["pilot_case_id"]:
        missing.append("pilot_case_id")

    if not normalized["question"]:
        missing.append("question")

    if not normalized["answer"]:
        missing.append("answer")

    if not normalized["evidence"]:
        missing.append("evidence")

    if missing:
        normalization_issues.append({
            "case": index,
            "pilot_case_id": normalized["pilot_case_id"],
            "missing": missing,
        })


question_sources = {}

for case in author_cases:
    source = case["question_source"] or "missing"
    question_sources[source] = question_sources.get(source, 0) + 1


normalization_checks = {
    "normalized_cases_22": len(author_cases) == EXPECTED_CASE_COUNT,
    "unique_pilot_ids_22": (
        len({
            case["pilot_case_id"]
            for case in author_cases
        }) == EXPECTED_CASE_COUNT
    ),
    "all_questions_present": all(
        bool(case["question"])
        for case in author_cases
    ),
    "all_answers_present": all(
        bool(case["answer"])
        for case in author_cases
    ),
    "all_evidence_present": all(
        bool(case["evidence"])
        for case in author_cases
    ),
    "no_normalization_issues": not normalization_issues,
}

normalization_gate = all(
    normalization_checks.values()
)

print("Normalized cases:", len(author_cases))
print("Question schema sources:", question_sources)
print("Normalization issues:", normalization_issues)

print("\nNORMALIZATION CHECKS")
for name, status in normalization_checks.items():
    print(f"{name}:", status)

print(
    "\nCODE CELL 03 GATE:",
    "PASS" if normalization_gate else "FAIL",
)

assert normalization_gate, (
    "One or more Author Review cases could not be normalized safely."
)

Normalized cases: 22
Question schema sources: {'accepted_model_draft': 4, 'question_review': 18}
Normalization issues: []

NORMALIZATION CHECKS
normalized_cases_22: True
unique_pilot_ids_22: True
all_questions_present: True
all_answers_present: True
all_evidence_present: True
no_normalization_issues: True

CODE CELL 03 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>نمایش بسته بازبینی مؤلف</h2>

<p>
در این بخش هر ۲۲ Case با یک قالب یکسان و کاملاً راست‌به‌چپ نمایش داده می‌شود.
تمرکز اصلی روی متن منبع، سؤال و پاسخ است و جزئیات فنی فقط در بخش بازشونده «مشخصات فنی» قرار می‌گیرند.
</p>

<p>
وضعیت <strong>«تأییدشده در بازبینی اولیه»</strong> فقط نشان می‌دهد Case از Human Review اولیه پروژه عبور کرده است؛
این وضعیت به معنی تأیید دکتر نیست.
برای مؤلف، وضعیت هر Case در شروع باید <strong>«هنوز بررسی نشده»</strong> باشد.
</p>

<p>
اگر دکتر با یک Case موافق نیست، کافی است شماره Case و نوع ایراد را مشخص کند:
«اصلاح سؤال»، «اصلاح پاسخ»، «اشکال در برداشت از منبع» یا «اشکال در استناد».
ثبت رسمی این تصمیم بعداً در Registry و با حفظ Lineage انجام می‌شود.
</p>

</div>

In [4]:
# Code Cell 04

def fa_status(value):
    mapping = {
        "ACCEPT": "تأییدشده در بازبینی اولیه",
        "NOT_STARTED": "هنوز توسط مؤلف بررسی نشده",
        "PENDING": "در انتظار بررسی",
        "REVISE": "نیازمند اصلاح",
        "REJECT": "رد شده",
    }

    if value is None:
        return "نامشخص"

    return mapping.get(
        str(value).strip().upper(),
        str(value),
    )


def evidence_locator(record):
    locator = record.get("locator") or {}

    if not isinstance(locator, dict):
        locator = {}

    parts = []

    paragraph_number = locator.get("paragraph_number")
    if paragraph_number is not None:
        parts.append(f"پاراگراف {paragraph_number}")

    line_start = locator.get("line_start")
    line_end = locator.get("line_end")

    if line_start is not None:
        if line_end is not None and line_end != line_start:
            parts.append(f"خطوط {line_start} تا {line_end}")
        else:
            parts.append(f"خط {line_start}")

    timestamp_start = locator.get("timestamp_start")
    timestamp_end = locator.get("timestamp_end")

    if timestamp_start is not None:
        if timestamp_end is not None:
            parts.append(
                f"زمان {timestamp_start} تا {timestamp_end}"
            )
        else:
            parts.append(
                f"زمان {timestamp_start}"
            )

    document_id = (
        record.get("document_id")
        or locator.get("paragraph_id")
        or locator.get("document_id")
    )

    if document_id:
        parts.append(
            f"شناسه منبع: {document_id}"
        )

    return " | ".join(parts)


def render_evidence(records):
    blocks = []

    for evidence_index, record in enumerate(
        records,
        start=1,
    ):
        text = html.escape(
            record.get("text", "").strip()
        )

        locator = html.escape(
            evidence_locator(record)
        )

        group = record.get(
            "_viewer_group",
            "primary",
        )

        group_label = {
            "primary": "شاهد اصلی",
            "supporting": "شاهد تکمیلی",
            "citation": "شاهد استنادی",
        }.get(
            group,
            "شاهد منبع",
        )

        locator_html = (
            f'<div class="locator">{locator}</div>'
            if locator
            else
            ""
        )

        blocks.append(
            f"""
            <div class="evidence-block">
                <div class="evidence-label">
                    {group_label} {evidence_index}
                </div>
                <div class="evidence-text">
                    {text.replace(chr(10), "<br>")}
                </div>
                {locator_html}
            </div>
            """
        )

    return "\n".join(blocks)


style = """
<style>
.author-review-root {
    direction: rtl;
    text-align: right;
    line-height: 1.95;
    font-family: Tahoma, Arial, sans-serif;
}
.author-summary {
    border: 1px solid #b8b8b8;
    border-radius: 12px;
    padding: 18px 20px;
    margin: 12px 0 24px 0;
}
.case-card {
    border: 1px solid #c9c9c9;
    border-radius: 14px;
    padding: 20px 22px;
    margin: 0 0 28px 0;
}
.case-title {
    font-size: 1.25rem;
    font-weight: 700;
    margin-bottom: 14px;
}
.section-title {
    font-weight: 700;
    margin-top: 18px;
    margin-bottom: 7px;
}
.question-box,
.answer-box,
.author-box {
    border-right: 4px solid #8a8a8a;
    padding: 10px 14px;
    margin: 8px 0;
}
.evidence-block {
    padding: 11px 13px;
    margin: 9px 0;
    border: 1px solid #dedede;
    border-radius: 9px;
}
.evidence-label {
    font-weight: 700;
    margin-bottom: 6px;
}
.evidence-text {
    white-space: normal;
}
.locator {
    direction: ltr;
    text-align: left;
    font-size: 0.82rem;
    margin-top: 8px;
    opacity: 0.78;
}
.status-line {
    margin-top: 8px;
}
details {
    margin-top: 14px;
}
code {
    direction: ltr;
    unicode-bidi: embed;
}
</style>
"""

summary_html = f"""
<div class="author-review-root">
    <div class="author-summary">
        <h2>بسته بازبینی مؤلف آماده است</h2>
        <p>
            تعداد Caseها: <strong>{len(author_cases)} از {EXPECTED_CASE_COUNT}</strong><br>
            پایلوت باقی‌مانده: <strong>{review_registry.get("remaining_pilot_count")}</strong><br>
            وضعیت بازبینی نهایی مؤلف:
            <strong>{html.escape(fa_status(author_review.get("final_review_status")))}</strong>
        </p>
        <p>
            لطفاً هر Case را فقط از نظر صحت علمی و معنایی سؤال، پاسخ و شواهد منبع بررسی کنید.
            تأیید بازبینی اولیه پروژه جایگزین نظر نهایی مؤلف نیست.
        </p>
    </div>
</div>
"""

display(
    HTML(
        style
        +
        summary_html
    )
)


for case in author_cases:
    question_html = html.escape(
        case["question"]
    ).replace(
        "\n",
        "<br>"
    )

    answer_html = html.escape(
        case["answer"]
    ).replace(
        "\n",
        "<br>"
    )

    citation_decision = fa_status(
        case["citation_decision"]
    )

    author_decision = fa_status(
        case["author_status"]
    )

    pilot_id = html.escape(
        str(
            case["pilot_case_id"]
        )
    )

    citation_id = html.escape(
        str(
            case["citation_mapping_id"]
            or "—"
        )
    )

    evidence_html = render_evidence(
        case["evidence"]
    )

    card = f"""
    <div class="author-review-root">
        <div class="case-card">

            <div class="case-title">
                مورد {case["number"]} از {EXPECTED_CASE_COUNT}
            </div>

            <div class="section-title">
                متن و شواهد منبع
            </div>

            {evidence_html}

            <div class="section-title">
                سؤال نهایی پیشنهادی
            </div>

            <div class="question-box">
                {question_html}
            </div>

            <div class="section-title">
                پاسخ نهایی پیشنهادی
            </div>

            <div class="answer-box">
                {answer_html}
            </div>

            <div class="section-title">
                وضعیت استناد
            </div>

            <div class="status-line">
                {html.escape(citation_decision)}
            </div>

            <div class="author-box">
                <strong>نظر نهایی مؤلف:</strong>
                {html.escape(author_decision)}
                <br>
                <span>
                    در صورت نیاز به اصلاح، شماره همین مورد و توضیح اصلاح پیشنهادی ثبت شود.
                </span>
            </div>

            <details>
                <summary>مشخصات فنی برای ردیابی</summary>
                <p>
                    Pilot ID:
                    <code>{pilot_id}</code>
                    <br>
                    Citation Mapping:
                    <code>{citation_id}</code>
                    <br>
                    Question source:
                    <code>{html.escape(str(case["question_source"]))}</code>
                </p>
            </details>

        </div>
    </div>
    """

    display(
        HTML(
            card
        )
    )


print(
    f"Rendered Author Review cases: {len(author_cases)} / {EXPECTED_CASE_COUNT}"
)

print(
    "CODE CELL 04 GATE:",
    "PASS" if len(author_cases) == EXPECTED_CASE_COUNT else "FAIL",
)

assert len(author_cases) == EXPECTED_CASE_COUNT

Rendered Author Review cases: 22 / 22
CODE CELL 04 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>اعتبارسنجی مستقل بسته بازبینی مؤلف</h2>

<p>
این مرحله مستقل از ظاهر Viewer بررسی می‌کند که هیچ Case در نمایش ۲۲تایی ناقص نشده باشد.
برای هر Pilot باید شناسه یکتا، سؤال نهایی، پاسخ نهایی و حداقل یک شاهد منبع موجود باشد.
همچنین Answer و Citation باید از Human Review اولیه عبور کرده باشند.
</p>

<p>
این Audit به‌خصوص برای جلوگیری از تکرار مشکل نسخه قبلی مهم است؛
نسخه قبلی ۲۲ Case را می‌خواند اما سؤال Caseهای Schema جدید را نمایش نمی‌داد.
PASS این مرحله یعنی هر ۲۲ Case اکنون از نظر محتوای لازم برای بازبینی مؤلف کامل هستند.
</p>

</div>

In [5]:
# Code Cell 05

def review_decision_is_accept(review):
    if not isinstance(review, dict):
        return False

    for key in [
        "human_decision",
        "decision",
        "human_adjudication_status",
    ]:
        if is_accept(review.get(key)):
            return True

    return False


audit_rows = []
audit_issues = []


for normalized in author_cases:
    raw_case = normalized["raw_case"]

    answer_review = (
        raw_case.get("expected_answer_review")
        or {}
    )

    citation_review = (
        raw_case.get("citation_review")
        or {}
    )

    answer_accepted = review_decision_is_accept(
        answer_review
    )

    citation_accepted = review_decision_is_accept(
        citation_review
    )

    question_valid = bool(
        normalized["question"]
    )

    answer_valid = bool(
        normalized["answer"]
    )

    evidence_valid = bool(
        normalized["evidence"]
    )

    row = {
        "number": normalized["number"],
        "pilot_case_id": normalized["pilot_case_id"],
        "question_present": question_valid,
        "answer_present": answer_valid,
        "evidence_present": evidence_valid,
        "answer_review_accept": answer_accepted,
        "citation_review_accept": citation_accepted,
    }

    audit_rows.append(row)

    if not all([
        question_valid,
        answer_valid,
        evidence_valid,
        answer_accepted,
        citation_accepted,
    ]):
        audit_issues.append(row)


author_packet_checks = {
    "exactly_22_cases": len(author_cases) == EXPECTED_CASE_COUNT,
    "exactly_22_unique_ids": (
        len({
            case["pilot_case_id"]
            for case in author_cases
        }) == EXPECTED_CASE_COUNT
    ),
    "all_22_questions_present": all(
        row["question_present"]
        for row in audit_rows
    ),
    "all_22_answers_present": all(
        row["answer_present"]
        for row in audit_rows
    ),
    "all_22_evidence_present": all(
        row["evidence_present"]
        for row in audit_rows
    ),
    "all_answers_human_accepted": all(
        row["answer_review_accept"]
        for row in audit_rows
    ),
    "all_citations_human_accepted": all(
        row["citation_review_accept"]
        for row in audit_rows
    ),
    "remaining_queue_zero": review_registry.get("remaining_pilot_count") == 0,
    "active_workspace_none": review_registry.get("active_workspace") is None,
    "no_author_packet_issues": not audit_issues,
}

author_packet_gate = all(
    author_packet_checks.values()
)

print("Author Review cases:", len(author_cases))
print("Cases with questions:", sum(row["question_present"] for row in audit_rows))
print("Cases with answers:", sum(row["answer_present"] for row in audit_rows))
print("Cases with evidence:", sum(row["evidence_present"] for row in audit_rows))
print("Cases with accepted answers:", sum(row["answer_review_accept"] for row in audit_rows))
print("Cases with accepted citations:", sum(row["citation_review_accept"] for row in audit_rows))
print("Issues:", audit_issues)

print("\nAUTHOR PACKET CHECKS")
for name, status in author_packet_checks.items():
    print(f"{name}:", status)

print(
    "\nAUTHOR REVIEW PACKET:",
    "READY" if author_packet_gate else "NOT_READY",
)

print(
    "CODE CELL 05 GATE:",
    "PASS" if author_packet_gate else "FAIL",
)

assert author_packet_gate, (
    "The 22-case Author Review packet is incomplete."
)

Author Review cases: 22
Cases with questions: 22
Cases with answers: 22
Cases with evidence: 22
Cases with accepted answers: 22
Cases with accepted citations: 22
Issues: []

AUTHOR PACKET CHECKS
exactly_22_cases: True
exactly_22_unique_ids: True
all_22_questions_present: True
all_22_answers_present: True
all_22_evidence_present: True
all_answers_human_accepted: True
all_citations_human_accepted: True
remaining_queue_zero: True
active_workspace_none: True
no_author_packet_issues: True

AUTHOR REVIEW PACKET: READY
CODE CELL 05 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Final Notebook Health</h2>

<p>
این Health Check خود فایل Notebook را از Disk می‌خواند و ساختار اجرایی آن را بررسی می‌کند:
شماره‌گذاری Code Cellها، اجراشدن Cellهای قبلی، نبود Error Output،
وجود Bootstrap مرکزی، نبود Import تکراری خارج از Bootstrap،
و وجود Gate بسته ۲۲ Case.
</p>

<p>
برای اینکه Health وضعیت واقعی فایل را ببیند، پیش از اجرای این Cell یک بار Notebook را Save کنید.
این Cell فقط Read-Only است و هیچ Artifact پروژه را تغییر نمی‌دهد.
</p>

</div>

In [6]:
# Code Cell 06

def find_current_notebook():
    expected_name = "12c_human_review_for_author.ipynb"
    cwd = Path.cwd().resolve()

    candidates = []

    for parent in [cwd, *cwd.parents]:
        candidates.extend([
            parent / expected_name,
            parent / "notebooks" / "phase2" / expected_name,
        ])

    seen = set()

    for candidate in candidates:
        candidate = candidate.resolve()
        key = str(candidate)

        if key in seen:
            continue

        seen.add(key)

        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"{expected_name} was not found. "
        "Save this notebook under notebooks/phase2 before running Final Health."
    )


NOTEBOOK_PATH = find_current_notebook()

disk_notebook = json.loads(
    NOTEBOOK_PATH.read_text(
        encoding="utf-8"
    )
)

disk_cells = disk_notebook.get(
    "cells",
    []
)

code_cell_pattern = re.compile(
    r"(?mi)^\s*#\s*Code Cell\s+(\d+)\b"
)

numbered_cells = []
error_cells_1_5 = []
imports_by_cell = {}


for notebook_index, cell in enumerate(
    disk_cells
):
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", "")

    if isinstance(source, list):
        source = "".join(source)

    match = code_cell_pattern.search(source)

    number = (
        int(match.group(1))
        if match
        else None
    )

    if number is not None:
        numbered_cells.append({
            "number": number,
            "execution_count": cell.get("execution_count"),
            "source": source,
            "notebook_index": notebook_index,
        })

    for line in source.splitlines():
        stripped = line.strip()

        if (
            stripped.startswith("import ")
            or stripped.startswith("from ")
        ):
            imports_by_cell.setdefault(
                number,
                []
            ).append(stripped)

    if number is None or number > 5:
        continue

    for output in cell.get("outputs", []) or []:
        if (
            isinstance(output, dict)
            and output.get("output_type") == "error"
        ):
            error_cells_1_5.append({
                "code_cell": number,
                "ename": output.get("ename"),
                "evalue": output.get("evalue"),
            })


numbers = [
    item["number"]
    for item in numbered_cells
]

missing_numbers = sorted(
    set(range(1, 7))
    -
    set(numbers)
)

duplicate_numbers = sorted(
    {
        number
        for number in numbers
        if numbers.count(number) > 1
    }
)

unexecuted_1_5 = [
    item["number"]
    for item in numbered_cells
    if (
        item["number"] <= 5
        and item["execution_count"] is None
    )
]

imports_outside_bootstrap = []

for number, import_lines in imports_by_cell.items():
    if number == 1:
        continue

    for line in import_lines:
        imports_outside_bootstrap.append({
            "code_cell": number,
            "import": line,
        })


all_source = "\n".join(
    "".join(cell.get("source", []))
    if isinstance(cell.get("source", []), list)
    else str(cell.get("source", ""))
    for cell in disk_cells
)

health_checks = {
    "code_cells_1_to_6_present": (
        numbers == [1, 2, 3, 4, 5, 6]
    ),
    "missing_code_cell_numbers_zero": not missing_numbers,
    "duplicate_code_cell_numbers_zero": not duplicate_numbers,
    "cells_1_to_5_executed": not unexecuted_1_5,
    "error_outputs_cells_1_to_5_zero": not error_cells_1_5,
    "bootstrap_is_code_cell_01": 1 in imports_by_cell,
    "repeated_imports_outside_bootstrap_zero": not imports_outside_bootstrap,
    "author_packet_gate_runtime_pass": globals().get("author_packet_gate") is True,
    "author_packet_marker_present": "AUTHOR REVIEW PACKET:" in all_source,
    "final_health_marker_present": "NOTEBOOK HEALTH:" in all_source,
}

notebook_health = all(
    health_checks.values()
)

print("Notebook:", NOTEBOOK_PATH)
print("Code cells:", len(numbered_cells))
print("Executed before Health:", len([
    item
    for item in numbered_cells
    if (
        item["number"] <= 5
        and item["execution_count"] is not None
    )
]))
print("Unexecuted Code Cells 1..5:", unexecuted_1_5)
print("Error cells 1..5:", error_cells_1_5)
print("Missing code-cell numbers:", missing_numbers)
print("Duplicate code-cell numbers:", duplicate_numbers)
print("Repeated imports outside bootstrap:", imports_outside_bootstrap)

print("\nFINAL HEALTH CHECKS")
for name, status in health_checks.items():
    print(f"{name}:", status)

print(
    "\nNOTEBOOK HEALTH:",
    "PASS" if notebook_health else "FAIL",
)

assert notebook_health, (
    "Author Review notebook health failed. "
    "Inspect the failed gate(s) above."
)

Notebook: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/12c_human_review_for_author.ipynb
Code cells: 6
Executed before Health: 5
Unexecuted Code Cells 1..5: []
Error cells 1..5: []
Missing code-cell numbers: []
Duplicate code-cell numbers: []
Repeated imports outside bootstrap: []

FINAL HEALTH CHECKS
code_cells_1_to_6_present: True
missing_code_cell_numbers_zero: True
duplicate_code_cell_numbers_zero: True
cells_1_to_5_executed: True
error_outputs_cells_1_to_5_zero: True
bootstrap_is_code_cell_01: True
repeated_imports_outside_bootstrap_zero: True
author_packet_gate_runtime_pass: True
author_packet_marker_present: True
final_health_marker_present: True

NOTEBOOK HEALTH: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>نتیجه‌گیری نهایی Notebook بازبینی مؤلف</h2>

<p>
هدف این Notebook ساخت یک نمای مستقل، خوانا و قابل‌ردیابی برای بازبینی علمی ۲۲ Pilot Golden Dataset توسط مؤلف بود.
منبع حقیقت داده‌ها <code>human_review_registry.json</code> باقی می‌ماند و خود Notebook هیچ Case را Hard-code یا بازنویسی نمی‌کند.
</p>

<p>
نسخه قبلی اگرچه ۲۲ Case را از Registry می‌خواند، اما به دلیل تفاوت Schema تاریخی،
سؤال نهایی بسیاری از Caseهای جدید را نمایش نمی‌داد.
در این نسخه، Schema قدیمی و جدید به‌صورت Read-Only Normalize می‌شوند
تا برای هر ۲۲ Case، متن منبع، سؤال نهایی، پاسخ نهایی، وضعیت استناد و وضعیت نظر مؤلف نمایش داده شود.
</p>

<p>
این Notebook قرار نیست به‌جای مؤلف تصمیم بگیرد.
وضعیت ACCEPT موجود در Caseها فقط مربوط به Human Review اولیه پروژه است.
نظر نهایی دکتر کمیل رودی یک Review مستقل است و تا زمانی که واقعاً انجام و در Workflow مجاز ثبت نشده باشد،
<code>NOT_STARTED</code> باقی می‌ماند.
</p>

<p>
اگر <code>CODE CELL 05 GATE: PASS</code> و <code>NOTEBOOK HEALTH: PASS</code> باشند،
بسته ۲۲ Case از نظر فنی برای ارائه به مؤلف آماده است.
این PASS به معنی تأیید علمی Caseها توسط مؤلف نیست؛ فقط کامل و سالم بودن بسته بازبینی را ثابت می‌کند.
</p>

<p>
<strong>NOTEBOOK OBJECTIVE: ACHIEVED</strong>
فقط در معنای «آماده‌سازی Viewer کامل و سالم برای Author Review».
خود <strong>Author Final Review</strong> تا دریافت نظر واقعی مؤلف همچنان انجام‌نشده باقی می‌ماند.
</p>

</div>